# Analysis histograms of the step2 graphs
2D heatmaps of the MC expectation in the analysis binning for every graph in `../graphs`
(best fit, 3-flavor best fit, and the detector systematic scans), with and without the SnowStorm gradient shift.
One figure per graph and variant with cascades, tracks and double cascades next to each other.
Bins with an MC expectation of 0 or below are counted per channel; they are drawn in grey (= 0) and hatched (< 0).

In [ ]:
import os, sys, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from NNMFit.utilities import load_pickle

STEP2_DIR = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/step2_hese_flavor_reunblind"
sys.path.insert(0, STEP2_DIR)
from plot_configs import _CHAN_SPECS  # analysis binning per channel

graphs_path = f"{STEP2_DIR}/graphs"
plotting_path = f"{STEP2_DIR}/studies/plots_analysis_histograms"
os.makedirs(plotting_path, exist_ok=True)

In [ ]:
graph_names = sorted(os.path.basename(d) for d in glob.glob(f"{graphs_path}/*") if os.path.exists(f"{d}/MC_Histogram.pickle"))
variants = {"syst": "MC_Histogram.pickle", "nosyst": "MC_Histogram_nosyst.pickle"}
print(f"{len(graph_names)} graphs")

In [ ]:
def to_2d(flat, spec):
    """Flat analysis histogram -> 2D array (energy, second dimension), same reshape as plot_functions.py;
    the second dimension is flipped where the 1D plots flip it (reco_zenith)."""
    dims = list(spec["binning"])
    h = np.reshape(flat, tuple(len(spec["binning"][d]) - 1 for d in dims))
    if spec["dim_info"][dims[1]]["flip"]:
        h = np.flip(h, axis=1)
    return h

def plot_graph(name, variant):
    hists = load_pickle(f"{graphs_path}/{name}/{variants[variant]}")["histograms"]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
    counts = {}
    for ax, (chan_key, chan_label, spec) in zip(axes, _CHAN_SPECS):
        dims = list(spec["binning"])
        xbins, ybins = spec["binning"][dims[0]], spec["binning"][dims[1]]
        h = to_2d(hists[spec["det_config"]], spec)
        n_zero, n_neg = int(np.sum(h == 0)), int(np.sum(h < 0))
        counts[chan_key] = {"bins": h.size, "zero": n_zero, "negative": n_neg, "min": h.min(), "total": h.sum()}

        positive = np.where(h > 0, h, np.nan)
        vmin = np.nanmin(positive) if np.any(h > 0) else 1e-6
        mesh = ax.pcolormesh(xbins, ybins, positive.T, norm=mcolors.LogNorm(vmin=vmin, vmax=np.nanmax(positive)), cmap="viridis")
        # bins with expectation 0 (grey) or below 0 (grey, hatched)
        for mask, hatch in [(h == 0, None), (h < 0, "xx")]:
            if mask.any():
                ax.pcolor(xbins, ybins, np.where(mask, 1.0, np.nan).T, cmap=mcolors.ListedColormap(["lightgrey"]),
                          hatch=hatch, edgecolor="k" if hatch else None, lw=0)
        fig.colorbar(mesh, ax=ax, label="MC expectation")

        ax.set_xscale("log" if spec["dim_info"][dims[0]]["log_x"] else "linear")
        ax.set_xlabel(spec["dim_info"][dims[0]]["x_label"])
        ax.set_ylabel(spec["dim_info"][dims[1]]["x_label"])
        ax.set_title(f"{chan_label}: {h.sum():.2f} events, {n_zero} bins = 0, {n_neg} bins < 0", fontsize=11)
    fig.suptitle(f"{name} ({variant})", fontsize=12)
    fig.savefig(f"{plotting_path}/{name}_{variant}.png", bbox_inches="tight")
    return fig, counts

In [ ]:
rows = []
for name in graph_names:
    for variant in variants:
        fig, counts = plot_graph(name, variant)
        plt.show() if name == graph_names[0] else plt.close(fig)  # only show the first graph inline
        for chan_key, c in counts.items():
            rows.append({"graph": name.replace("nbestfit50_all_param_2sigma_1D_10steps_PassingHESE", "bestfit"),
                         "variant": variant, "channel": chan_key, **c})
df_counts = pd.DataFrame(rows)
df_counts.to_csv(f"{plotting_path}/bin_counts.csv", index=False)

## Bins with MC expectation ≤ 0

In [ ]:
df_bins = df_counts.pivot_table(index=["graph", "variant"], columns="channel", values=["zero", "negative"], aggfunc="first")
df_bins = df_bins.reindex(columns=["cascades", "tracks", "dc"], level=1)
with pd.option_context("display.max_rows", 200, "display.width", 200, "display.max_columns", 20):
    display(df_bins)

In [ ]:
# most negative bin per graph and channel (with systematics)
df_min = df_counts[df_counts["variant"] == "syst"].pivot_table(index="graph", columns="channel", values="min", aggfunc="first")
with pd.option_context("display.max_rows", 200, "display.float_format", "{:.3g}".format):
    display(df_min[["cascades", "tracks", "dc"]])